# University GPU server edition

Run cells from top to bottom in a **fresh kernel for each notebook**, inside a GPU allocation.
See `README.md` in this folder for setup and known limitations. This copy does not install
packages, replace PyTorch, mount Drive, or modify the original checkpoint directory.
Outputs go to `GPU folders/outputs` (override with `DOOM_OUTPUT_ROOT` before starting Jupyter).
Training settings and evaluation episode counts retain the original defaults: these are full runs.
For direct SNN training, set `SMOKE_TEST = True` first for a short training check.


In [1]:
import os, sys, json, platform
from pathlib import Path
from importlib import metadata

# Works when Jupyter starts in either the repository or GPU folders.
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "doom_plaincnn_sanity_check.ipynb").is_file()
                  and (p / "GPU folders").is_dir()
                  and (p / "conversion_friendly_v1").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError("Start Jupyter from the doom-on-snn repository or GPU folders.")
WORK_DIR = REPO_ROOT / "GPU folders"
OUTPUT_ROOT = Path(os.environ.get("DOOM_OUTPUT_ROOT", str(WORK_DIR / "outputs"))).expanduser().resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
# Match worker count to the CPU allocation; allow an explicit lower limit.
CPU_BUDGET = int(os.environ.get("SLURM_CPUS_PER_TASK", len(os.sched_getaffinity(0))))
SERVER_WORKERS = max(1, min(8, int(os.environ.get("DOOM_NUM_WORKERS", max(1, CPU_BUDGET - 2))), CPU_BUDGET))
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
import numpy as np
import torch
import vizdoom, sample_factory
torch.set_num_threads(1)
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU visible. Run the Jupyter kernel inside your cluster GPU allocation.")
print("Python:", sys.version, "\nKernel:", sys.executable)
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU calculation:", (torch.ones(1, device="cuda") + 1).item())
print("CPU budget:", CPU_BUDGET, "workers:", SERVER_WORKERS)
print("Repository:", REPO_ROOT, "\nOutputs:", OUTPUT_ROOT)
versions = {p: metadata.version(p) for p in
            ["torch", "numpy", "sample-factory", "vizdoom", "gymnasium", "matplotlib"]}
(OUTPUT_ROOT / "environment.json").write_text(json.dumps({
    "python": sys.version, "executable": sys.executable, "packages": versions,
    "gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda,
}, indent=2))


Python: 3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0] 
Kernel: /usr/bin/python3
PyTorch: 2.14.1+cu130 CUDA: 13.0
GPU: NVIDIA RTX 6000 Ada Generation
GPU calculation: 2.0
CPU budget: 64 workers: 8
Repository: /home/e22405/doom-on-snn 
Outputs: /home/e22405/doom-on-snn/GPU folders/outputs


339

# Directly trained SNN on ViZDoom `doom_basic` (Sample Factory + surrogate gradients)

Trains a spiking policy **from scratch with reinforcement learning**, so it can be compared with the
**converted** SNN (threshold-balanced from the ANN) on the same scenario.

**What is trained.** Same layer sizes as the ANN baseline (`convnet_simple`): Conv(3→32, k8, s4) → Conv(32→64, k4, s2) →
Conv(64→128, k3, s2) → Linear(2304→512), then a linear action head and a linear value head. Every hidden layer is an
integrate-and-fire layer (threshold 1, soft reset, initial membrane 0.5, direct analog input every step) run for
`T` steps on each frame and trained end to end with a triangular surrogate gradient. The 512-layer's mean firing
rate (× a fixed scale) feeds the heads, so the head applied to the rate equals the "accumulate the output over T steps"
readout used for the converted SNN. State is reset every frame: no recurrence, no frame stacking.

**Protocol identical to the converted-SNN evaluation:** same `ObservationNormalizer` (statistics learned during
training), deterministic argmax, one env step per decision (`ACTION_REPEAT = 1`), 300 episodes, same operation counting.

**Status of the code.** The spiking encoder was tested outside ViZDoom: forward pass identical to the converted-SNN loop,
gradients reach every layer, calibration hits its target rates, and Sample Factory trained it end to end on a toy image task (checkpoint
reload and standalone evaluator match Sample Factory's own model exactly). It has **not** been run on ViZDoom yet, so
do a short smoke test first (`SMOKE_TEST = True`).

Run on the Linux server inside a **GPU allocation**. Edit only the settings cell to change `T` or the seed.

### 1. Use the existing server environment

Dependencies are checked in the first code cell. No administrator commands are run.


In [2]:
# Existing environment validated in server setup; no system package installation.


Continue in the same kernel. No packages were installed by this notebook.


### 2. GPU check and the `torch.load` fix (safe to re-run)

In [3]:
# Sample Factory also calls torch.load internally. Its trusted training checkpoints
# contain NumPy metadata. Preserve explicit weights_only arguments; patch only once.
# Only use checkpoints you produced or obtained from a trusted source.
if not getattr(torch.load, "_doom_trusted_load", False):
    def _trusted_load(*args, _load=torch.load, **kwargs):
        kwargs.setdefault("weights_only", False)
        return _load(*args, **kwargs)
    _trusted_load._doom_trusted_load = True
    torch.load = _trusted_load
print("Trusted checkpoint loading enabled; models and inputs still run on CUDA.")


Trusted checkpoint loading enabled; models and inputs still run on CUDA.


### 3. Settings
Change only this cell to run another `T` or seed. `SMOKE_TEST = True` trains for 100k steps to check the pipeline and measure
speed before committing to the full run.

In [4]:
import os

ENV_NAME = "doom_basic"
SNN_T = 8                  # time steps per frame during training (the converted SNN needed T=32)
SEED = 0
SMOKE_TEST = False          # True: 100k steps just to test the pipeline. Set False for the real 2M-step run.

TOTAL_STEPS = 100_000 if SMOKE_TEST else 8_000_000
NUM_WORKERS, ENVS_PER_WORKER = SERVER_WORKERS, 4
BATCH_SIZE = 1024          # lower to 512 if the GPU runs out of memory (memory grows with SNN_T)
TARGET_RATE = 0.15         # mean firing rate per layer the calibration aims for at initialisation
FEAT_SCALE = 4.0           # fixed scale on the mean firing rate that feeds the heads
ACTION_REPEAT = 1          # keep identical to the converted-SNN evaluation
N_EVAL_EPISODES = 300

EXPERIMENT = f"snn_direct_T{SNN_T}_seed{SEED}" + ("_smoke" if SMOKE_TEST else "")
TRAIN_DIR = str(OUTPUT_ROOT / "train_dir")
RESULTS_DIR = str(OUTPUT_ROOT / "direct_snn_results")
os.makedirs(RESULTS_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("experiment:", EXPERIMENT, "| device:", device)

experiment: snn_direct_T8_seed0 | device: cuda


### 4. The spiking encoder module
Written to `snn_direct.py` so it can be reused and kept in the repository. It contains the surrogate-gradient neuron, the
spiking layers (`SpikingTrunk`), the Sample Factory encoder wrapper, the gain calibration, and a standalone evaluator
that loads a checkpoint without Sample Factory's model classes.

In [5]:
%%writefile snn_direct.py
"""Directly trained spiking policy for Sample Factory (ViZDoom doom_basic).

Same layer sizes as the ANN baseline (convnet_simple): 3 convs + a 512 layer, but every
hidden layer is an integrate-and-fire (IF) layer unrolled over T time steps and trained
with surrogate gradients. State is reset for every frame (no recurrence across frames).
"""
import numpy as np
import torch
from torch import nn

from sample_factory.algo.utils.torch_utils import calc_num_elements
from sample_factory.model.encoder import Encoder


# ----------------------------------------------------------------------------------------
# Neuron: Heaviside spike in the forward pass, triangular surrogate in the backward pass
# ----------------------------------------------------------------------------------------
class SpikeFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):  # x = membrane - threshold
        ctx.save_for_backward(x)
        return (x >= 0).to(x.dtype)

    @staticmethod
    def backward(ctx, grad_out):
        (x,) = ctx.saved_tensors
        return grad_out * torch.clamp(1.0 - x.abs(), min=0.0)  # triangular, width 1


def if_step(v, current, threshold=1.0, v_init=0.5, detach_reset=True):
    """One integrate-and-fire step, soft reset (subtract threshold).

    Forward behaviour is identical to the if_step used for the converted SNN."""
    if v is None:
        v = torch.full_like(current, v_init * threshold)
    v = v + current
    s = SpikeFn.apply(v - threshold)
    v = v - (s.detach() if detach_reset else s) * threshold
    return v, s


# ----------------------------------------------------------------------------------------
# The spiking trunk (no Sample Factory dependency in the maths, so it can be evaluated alone)
# ----------------------------------------------------------------------------------------
class SpikingTrunk(nn.Module):
    def __init__(self, in_channels=3, obs_hw=(72, 128), T=8, v_init=0.5,
                 gains=(1.0, 1.0, 1.0, 1.0), feat_scale=4.0):
        super().__init__()
        self.T = int(T)
        self.v_init = float(v_init)
        self.conv1 = nn.Conv2d(in_channels, 32, 8, stride=4)
        self.conv2 = nn.Conv2d(32, 64, 4, stride=2)
        self.conv3 = nn.Conv2d(64, 128, 3, stride=2)
        # local (unregistered) stack, only used to compute the flattened size; keeps state_dict keys unique
        self.flat_size = calc_num_elements(nn.Sequential(self.conv1, self.conv2, self.conv3), (in_channels, *obs_hw))
        self.fc = nn.Linear(self.flat_size, 512)
        # fixed (non-trainable) per-layer current gains and output scale: buffers, so they are saved
        # in the checkpoint and are not touched by Sample Factory's weight initialisation
        self.register_buffer("gains", torch.tensor(list(gains), dtype=torch.float32))
        self.register_buffer("feat_scale", torch.tensor(float(feat_scale)))
        self.out_size = 512

    def forward(self, x, T=None, return_spikes=False):
        T = self.T if T is None else int(T)
        g = self.gains
        c1 = self.conv1(x) * g[0]  # direct encoding: identical every step, computed once
        v1 = v2 = v3 = v4 = None
        acc = 0
        spikes = [0.0, 0.0, 0.0, 0.0]
        for _ in range(T):
            v1, s1 = if_step(v1, c1, v_init=self.v_init)
            v2, s2 = if_step(v2, self.conv2(s1) * g[1], v_init=self.v_init)
            v3, s3 = if_step(v3, self.conv3(s2) * g[2], v_init=self.v_init)
            v4, s4 = if_step(v4, self.fc(s3.flatten(1)) * g[3], v_init=self.v_init)
            acc = acc + s4
            if return_spikes:
                for k, s in enumerate((s1, s2, s3, s4)):
                    spikes[k] = spikes[k] + s.detach().flatten(1).sum(1)  # per-sample spike count
        feats = acc * (self.feat_scale / T)  # mean firing rate of the 512 layer, scaled
        if return_spikes:
            return feats, torch.stack(spikes, 1)  # (B, 4) spikes per decision per layer
        return feats


# ----------------------------------------------------------------------------------------
# Sample Factory glue
# ----------------------------------------------------------------------------------------
def add_snn_args(parser):
    p = parser
    p.add_argument("--snn_T", default=8, type=int, help="Time steps per frame for the spiking encoder")
    p.add_argument("--snn_v_init", default=0.5, type=float, help="Initial membrane potential / threshold")
    p.add_argument("--snn_gains", default="1,1,1,1", type=str,
                   help="Comma separated fixed current gains for the 4 spiking layers (from calibration)")
    p.add_argument("--snn_feat_scale", default=4.0, type=float, help="Scale on the mean firing rate fed to the heads")


class SpikingEncoder(Encoder):
    def __init__(self, cfg, obs_space):
        super().__init__(cfg)
        obs_shape = obs_space["obs"].shape
        gains = [float(v) for v in str(cfg.snn_gains).split(",")]
        assert len(gains) == 4, "--snn_gains needs 4 comma separated numbers"
        self.trunk = SpikingTrunk(obs_shape[0], tuple(obs_shape[1:]), cfg.snn_T, cfg.snn_v_init,
                                  gains, cfg.snn_feat_scale)
        self.encoder_out_size = self.trunk.out_size

    def forward(self, obs_dict):
        return self.trunk(obs_dict["obs"])

    def get_out_size(self):
        return self.encoder_out_size


def make_snn_encoder(cfg, obs_space):
    return SpikingEncoder(cfg, obs_space)


# ----------------------------------------------------------------------------------------
# Data-dependent gain calibration (so the network does not start silent)
# ----------------------------------------------------------------------------------------
@torch.no_grad()
def calibrate_gains(frames, T=8, target_rate=0.15, v_init=0.5, iters=18, device="cpu", seed=0):
    """Pick fixed per-layer current gains so each layer's mean firing rate is ~target_rate at
    initialisation, using real (already standardised) frames. Weights use the same init Sample
    Factory applies (orthogonal, gain 1, zero bias)."""
    torch.manual_seed(seed)
    trunk = SpikingTrunk(frames.shape[1], tuple(frames.shape[2:]), T, v_init).to(device)
    for m in (trunk.conv1, trunk.conv2, trunk.conv3, trunk.fc):
        nn.init.orthogonal_(m.weight, gain=1.0)
        nn.init.zeros_(m.bias)
    x = frames.to(device)
    gains = [1.0, 1.0, 1.0, 1.0]

    def rate_of_layer(k, g):
        gg = list(gains)
        gg[k] = g
        c1 = trunk.conv1(x) * gg[0]
        v = [None] * 4
        total, count = 0.0, 0
        for _ in range(T):
            v[0], s1 = if_step(v[0], c1, v_init=v_init)
            if k == 0:
                total += s1.sum().item(); count += s1.numel(); continue
            v[1], s2 = if_step(v[1], trunk.conv2(s1) * gg[1], v_init=v_init)
            if k == 1:
                total += s2.sum().item(); count += s2.numel(); continue
            v[2], s3 = if_step(v[2], trunk.conv3(s2) * gg[2], v_init=v_init)
            if k == 2:
                total += s3.sum().item(); count += s3.numel(); continue
            v[3], s4 = if_step(v[3], trunk.fc(s3.flatten(1)) * gg[3], v_init=v_init)
            total += s4.sum().item(); count += s4.numel()
        return total / count

    for k in range(4):
        lo, hi = -4.0, 4.0  # search log10(gain)
        for _ in range(iters):
            mid = 0.5 * (lo + hi)
            if rate_of_layer(k, 10 ** mid) < target_rate:
                lo = mid
            else:
                hi = mid
        gains[k] = float(10 ** (0.5 * (lo + hi)))
    rates = [rate_of_layer(k, gains[k]) for k in range(4)]
    return gains, rates


# ----------------------------------------------------------------------------------------
# Standalone evaluator (no Sample Factory model), mirrors the PlainCNN / converted-SNN protocol
# ----------------------------------------------------------------------------------------
class DirectSNN(nn.Module):
    def __init__(self, T=8, v_init=0.5, gains=(1, 1, 1, 1), feat_scale=4.0, n_actions=4):
        super().__init__()
        self.trunk = SpikingTrunk(3, (72, 128), T, v_init, gains, feat_scale)
        self.head = nn.Linear(512, n_actions)

    def forward(self, x, T=None, return_spikes=False):
        out = self.trunk(x, T=T, return_spikes=return_spikes)
        if return_spikes:
            feats, spk = out
            return self.head(feats), spk
        return self.head(out)


def load_direct_snn(sf_state_dict, T, v_init=0.5, n_actions=4):
    """Build the standalone SNN from a Sample Factory checkpoint['model'] state dict."""
    gains = sf_state_dict["encoder.trunk.gains"].cpu().tolist()
    feat_scale = float(sf_state_dict["encoder.trunk.feat_scale"])
    net = DirectSNN(T, v_init, gains, feat_scale, n_actions)
    state = {}
    for k, v in sf_state_dict.items():
        if k.startswith("encoder.trunk."):
            state["trunk." + k[len("encoder.trunk."):]] = v
    state["head.weight"] = sf_state_dict["action_parameterization.distribution_linear.weight"]
    state["head.bias"] = sf_state_dict["action_parameterization.distribution_linear.bias"]
    missing, unexpected = net.load_state_dict(state, strict=False)
    return net.eval(), missing, unexpected

Overwriting snn_direct.py


### 5. Register the ViZDoom environments and the spiking encoder with Sample Factory

In [6]:
import copy, functools, glob, importlib, json, shutil, time
import numpy as np
import torch.nn as nn
import gymnasium as gym

import snn_direct as sd
importlib.reload(sd)

from sample_factory.algo.utils.context import global_model_factory
from sample_factory.cfg.arguments import parse_full_cfg, parse_sf_args
from sample_factory.envs.create_env import create_env
from sample_factory.envs.env_utils import register_env
from sample_factory.model.actor_critic import create_actor_critic
from sample_factory.train import run_rl
from sample_factory.utils.normalize import ObservationNormalizer
from sf_examples.vizdoom.doom.doom_params import add_doom_env_args, doom_override_defaults
from sf_examples.vizdoom.doom.doom_utils import DOOM_ENVS, make_doom_env_from_spec


def register_vizdoom_envs():
    for env_spec in DOOM_ENVS:
        register_env(env_spec.name, functools.partial(make_doom_env_from_spec, env_spec))


def parse_snn_doom_cfg(argv, evaluation=False):
    parser, _ = parse_sf_args(argv=argv, evaluation=evaluation)
    add_doom_env_args(parser)
    doom_override_defaults(parser)
    sd.add_snn_args(parser)
    return parse_full_cfg(parser, argv)


register_vizdoom_envs()
global_model_factory().register_encoder_factory(sd.make_snn_encoder)
print("registered ViZDoom envs and the spiking encoder")

/home/e22405/.local/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
[2026-11-02 18:08:19,013][3574448] register_encoder_factory: <function make_snn_encoder at 0x7ed1b0f7de40>


registered ViZDoom envs and the spiking encoder


### 6. Calibrate the starting gains
A spiking layer with the wrong scale starts silent (no spikes, so no gradient) or saturated. Before training, fixed per-layer
current gains are chosen so every layer fires at about `TARGET_RATE` on real frames. The gains are not trained; they are
saved in the checkpoint and in `config.json`.

The environment here is built the same way as in the PlainCNN / converted-SNN notebooks (a non-existent experiment name, so the
defaults are used), so the evaluation protocol is identical.

In [7]:
# New experiments only: do not recalibrate when resuming a trained checkpoint.
import random
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

env_cfg = parse_snn_doom_cfg([f"--env={ENV_NAME}", "--experiment=eval_env_only", f"--train_dir={TRAIN_DIR}", f"--seed={SEED}"], evaluation=True)
env = create_env(ENV_NAME, cfg=env_cfg, env_config=None)
print("observation space:", env.observation_space, "| action space:", env.action_space)

# frames from a random policy (a stand-in for early-training experience)
env.action_space.seed(SEED)
raw_frames = []
episode_index = 0
while len(raw_frames) < 800:
    obs, _ = env.reset(seed=SEED + episode_index)
    episode_index += 1
    done = truncated = False
    t = 0
    while not (done or truncated) and len(raw_frames) < 800:
        raw = obs["obs"] if isinstance(obs, dict) else obs
        if t % 2 == 0:
            raw_frames.append(np.asarray(raw))
        obs, r, done, truncated, _ = env.step(env.action_space.sample())
        t += 1
env.close()

# standardise like Sample Factory's normaliser: /255, then per-pixel (x - mean) / sqrt(var + eps), clip to +-5
x = torch.tensor(np.stack(raw_frames)).float() / 255.0
x = ((x - x.mean(0, keepdim=True)) / torch.sqrt(x.var(0, unbiased=False, keepdim=True) + 1e-5)).clamp(-5, 5)
print("calibration frames:", tuple(x.shape))

gains, rates = sd.calibrate_gains(x, T=SNN_T, target_rate=TARGET_RATE, device=device, seed=SEED)
print("gains:", [round(g, 4) for g in gains])
print("mean firing rate per layer at initialisation:", [round(r, 3) for r in rates])
# Retain exact calibration inputs and gains to audit/reuse this initialization.
calibration_dir = OUTPUT_ROOT / "calibration" / EXPERIMENT
calibration_dir.mkdir(parents=True, exist_ok=True)
torch.save({"frames": x.cpu(), "gains": gains, "rates": rates,
            "seed": SEED, "snn_T": SNN_T}, calibration_dir / "initialization.pt")
print("Calibration artifact:", calibration_dir / "initialization.pt")


[2026-11-02 18:08:19,074][3574448] Doom resolution: 160x120, resize resolution: (128, 72)


observation space: Box(0, 255, (3, 72, 128), uint8) | action space: Discrete(4)


[W][11575.107077] pw.conf      | [          conf.c: 1031 try_load_conf()] can't load config client-rt.conf: No such file or directory
[E][11575.107154] pw.conf      | [          conf.c: 1060 pw_conf_load_conf_for_context()] can't load config client-rt.conf: No such file or directory
[ALSOFT] (EE) Failed to create PipeWire event context (errno: 2)


calibration frames: (800, 3, 72, 128)
gains: [0.5001, 1.1513, 1.2109, 1.3703]
mean firing rate per layer at initialisation: [0.15, 0.15, 0.15, 0.15]
Calibration artifact: /home/e22405/doom-on-snn/GPU folders/outputs/calibration/snn_direct_T8_seed0/initialization.pt


### 7. Train

In [8]:
gains_arg = ",".join(f"{g:.6f}" for g in gains)
train_argv = [
    "--device=gpu", "--set_workers_cpu_affinity=False",
    f"--env={ENV_NAME}", f"--experiment={EXPERIMENT}", f"--train_dir={TRAIN_DIR}", f"--seed={SEED}",
    f"--num_workers={NUM_WORKERS}", f"--num_envs_per_worker={ENVS_PER_WORKER}", f"--batch_size={BATCH_SIZE}",
    f"--train_for_env_steps={TOTAL_STEPS}", "--nonlinearity=relu", "--use_rnn=False",
    f"--snn_T={SNN_T}", f"--snn_gains={gains_arg}", f"--snn_feat_scale={FEAT_SCALE}",
]
print(" ".join(train_argv))
train_cfg = parse_snn_doom_cfg(train_argv)
t0 = time.time()
status = run_rl(train_cfg)
print(f"training finished (status {status}) in {(time.time() - t0) / 60:.1f} min")

[2026-11-02 18:08:29,853][3574448] Loading existing experiment configuration from /home/e22405/doom-on-snn/GPU folders/outputs/train_dir/snn_direct_T8_seed0/config.json
[2026-11-02 18:08:29,854][3574448] Overriding arg 'train_for_env_steps' with value 8000000 passed from command line
[2026-11-02 18:08:29,861][3574448] Experiment dir /home/e22405/doom-on-snn/GPU folders/outputs/train_dir/snn_direct_T8_seed0 already exists!
[2026-11-02 18:08:29,862][3574448] Resuming existing experiment from /home/e22405/doom-on-snn/GPU folders/outputs/train_dir/snn_direct_T8_seed0...
[2026-11-02 18:08:29,863][3574448] Weights and Biases integration disabled


--device=gpu --set_workers_cpu_affinity=False --env=doom_basic --experiment=snn_direct_T8_seed0 --train_dir=/home/e22405/doom-on-snn/GPU folders/outputs/train_dir --seed=0 --num_workers=8 --num_envs_per_worker=4 --batch_size=1024 --train_for_env_steps=8000000 --nonlinearity=relu --use_rnn=False --snn_T=8 --snn_gains=0.500128,1.151338,1.210918,1.370331 --snn_feat_scale=4.0


[2026-11-02 18:08:32,415][3574448] Queried available GPUs: 0,1,2

[2026-11-02 18:08:32,417][3574448] Environment var CUDA_VISIBLE_DEVICES is 0,1,2

/home/e22405/.local/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
[2026-11-02 18:08:35,022][3574872] Doom resolution: 160x120, resize resolution: (128, 72)
/home/e22405/.local/lib/python3.12/site-packages/gymnasium/core.py:311: UserWarning: WARN: env.num_agents to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.num_agents` for environment variables or `env.get_wrapper_attr('num_agents')` that will search the reminding wrappers.
  logger.warn(
/home/e22405/.local/lib/python3.12/site-packages/gymnasium/core.py:311: UserWarning: WARN: env.is_multiagent to get variables from other wrappers is deprecated and will be removed in v1.0, to get this 

training finished (status 0) in 17.8 min


While it trains, watch the `Avg episode reward` lines. For reference, the ANN reached about 0.75 to 0.8 after 2M steps.
If the reward is still at the random-policy level after a few hundred thousand steps, the first things to change are `FEAT_SCALE`
(try 8 or 16) and the learning rate (`--learning_rate`), then `TARGET_RATE`. Report whatever you change.

In [9]:
print("Checkpoints and logs persist in", Path(TRAIN_DIR) / EXPERIMENT)


Checkpoints and logs persist in /home/e22405/doom-on-snn/GPU folders/outputs/train_dir/snn_direct_T8_seed0


### 8. Load the trained network and verify it
Rebuilds a standalone spiking network from the checkpoint and checks it against Sample Factory's own actor-critic on the same
frames (the logits must match exactly). After a kernel restart, rerun setup and registration before loading the persistent run folder.

In [10]:
run_dir = f"{TRAIN_DIR}/{EXPERIMENT}"
checkpoints = sorted(glob.glob(f"{run_dir}/checkpoint_p0/checkpoint_*.pth"))
if not checkpoints:
    raise FileNotFoundError(f"No checkpoints in {run_dir}. Run training first or select an existing experiment.")
ckpt_path = checkpoints[-1]
ckpt = torch.load(ckpt_path, map_location="cpu")
sf_state = ckpt["model"]
print("checkpoint:", os.path.basename(ckpt_path), "| env_steps:", ckpt["env_steps"])

saved_cfg = parse_snn_doom_cfg([f"--env={ENV_NAME}", f"--experiment={EXPERIMENT}", f"--train_dir={TRAIN_DIR}"], evaluation=True)
from sample_factory.cfg.arguments import load_from_checkpoint
saved_cfg = load_from_checkpoint(saved_cfg)
assert saved_cfg.use_rnn is False and saved_cfg.snn_gains != "1,1,1,1", "saved config was not loaded"
print("saved config | snn_T:", saved_cfg.snn_T, "| gains:", saved_cfg.snn_gains, "| feat_scale:", saved_cfg.snn_feat_scale,
      "| seed:", saved_cfg.seed)

net, missing, unexpected = sd.load_direct_snn(sf_state, T=saved_cfg.snn_T, v_init=saved_cfg.snn_v_init, n_actions=4)
print("standalone load | missing:", missing, "| unexpected:", unexpected)

obs_space = gym.spaces.Dict({"obs": gym.spaces.Box(low=0, high=255, shape=(3, 72, 128), dtype=np.uint8)})
ac = create_actor_critic(saved_cfg, obs_space, gym.spaces.Discrete(4))
print("strict load into Sample Factory's actor-critic:", ac.load_state_dict(sf_state, strict=True))
ac.eval()

xs = (x[:64] if "x" in globals() else torch.randn(64, 3, 72, 128).clamp(-5, 5)).cpu()
with torch.no_grad():
    sf_logits = ac.action_parameterization.distribution_linear(ac.encoder({"obs": xs}))
    my_logits = net.cpu()(xs)
d = (sf_logits - my_logits).abs().max().item()
print(f"max |logit difference|, Sample Factory vs standalone SNN: {d:.2e} ->", "PASS" if d < 1e-5 else "FAIL")
net = net.to(device).eval()

# the same normaliser (trained statistics) the converted-SNN evaluation used
normalizer = ObservationNormalizer(obs_space, saved_cfg)
norm_state = {k[len("obs_normalizer."):]: v for k, v in sf_state.items() if k.startswith("obs_normalizer.")}
print("normaliser load:", normalizer.load_state_dict(norm_state, strict=False))
normalizer.eval()

[2026-11-02 18:26:20,794][3574448] Loading existing experiment configuration from /home/e22405/doom-on-snn/GPU folders/outputs/train_dir/snn_direct_T8_seed0/config.json
[2026-11-02 18:26:20,795][3574448] Adding new argument 'no_render'=False that is not in the saved config file!
[2026-11-02 18:26:20,795][3574448] Adding new argument 'save_video'=False that is not in the saved config file!
[2026-11-02 18:26:20,795][3574448] Adding new argument 'video_frames'=1000000000.0 that is not in the saved config file!
[2026-11-02 18:26:20,796][3574448] Adding new argument 'video_name'=None that is not in the saved config file!
[2026-11-02 18:26:20,796][3574448] Adding new argument 'max_num_frames'=1000000000.0 that is not in the saved config file!
[2026-11-02 18:26:20,796][3574448] Adding new argument 'max_num_episodes'=1000000000.0 that is not in the saved config file!
[2026-11-02 18:26:20,796][3574448] Adding new argument 'push_to_hub'=False that is not in the saved config file!
[2026-11-02 18:

checkpoint: checkpoint_000001955_8007680.pth | env_steps: 8007680
saved config | snn_T: 8 | gains: 0.500128,1.151338,1.210918,1.370331 | feat_scale: 4.0 | seed: 0
standalone load | missing: [] | unexpected: []
strict load into Sample Factory's actor-critic: <All keys matched successfully>


/home/e22405/.local/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
[2026-11-02 18:26:21,105][3574448] RunningMeanStd input shape: (3, 72, 128)


max |logit difference|, Sample Factory vs standalone SNN: 0.00e+00 -> PASS
normaliser load: <All keys matched successfully>


ObservationNormalizer(
  (running_mean_std): RunningMeanStdDictInPlace(
    (running_mean_std): ModuleDict(
      (obs): RunningMeanStdInPlace()
    )
  )
)

### 9. Evaluate: reward at the training T and at other T
Same loop as the ANN and the converted SNN: deterministic argmax, one env step per decision, 300 episodes.

In [11]:
env = create_env(ENV_NAME, cfg=env_cfg, env_config=None)

def prep(obs):
    raw = obs["obs"] if isinstance(obs, dict) else obs
    d = {"obs": torch.as_tensor(raw).float().cpu().unsqueeze(0)}
    return normalizer(d)["obs"]  # the normaliser RETURNS a new dict; it does not modify d in place

def run_episodes(policy, n_episodes=N_EVAL_EPISODES, action_repeat=ACTION_REPEAT, keep_frames=0):
    rewards, kept = [], []
    for ep in range(n_episodes):
        obs, _ = env.reset()
        done = truncated = False
        total = 0.0
        while not (done or truncated):
            xin = prep(obs)
            if len(kept) < keep_frames:
                kept.append(xin.squeeze(0))
            with torch.no_grad():
                action = policy(xin.to(device)).argmax(1).item()
            for _ in range(action_repeat):
                obs, r, done, truncated, _ = env.step(action)
                total += float(r)
                if done or truncated:
                    break
        rewards.append(total)
    return (np.array(rewards), torch.stack(kept) if kept else None)

def summarize(name, r):
    se = r.std() / np.sqrt(len(r))
    print(f"{name:12s} {r.mean():.3f} ± {r.std():.3f} (SE {se:.3f}) | below 0: {(r<0).sum()} | below 0.3: {(r<0.3).sum()} | n={len(r)}")

T_TRAIN = saved_cfg.snn_T
T_LIST = sorted({2, 4, 8, 16, 32, T_TRAIN})
game, eval_frames = {}, None
for T in T_LIST:
    r, fr = run_episodes(lambda xin, T=T: net(xin, T=T), keep_frames=500 if T == T_TRAIN else 0)
    game[T] = r
    if fr is not None:
        eval_frames = fr
    summarize(f"direct T={T}" + (" (trained)" if T == T_TRAIN else ""), r)

[W][12657.155509] pw.conf      | [          conf.c: 1031 try_load_conf()] can't load config client-rt.conf: No such file or directory
[E][12657.155575] pw.conf      | [          conf.c: 1060 pw_conf_load_conf_for_context()] can't load config client-rt.conf: No such file or directory
[ALSOFT] (EE) Failed to create PipeWire event context (errno: 2)


direct T=2   0.804 ± 0.090 (SE 0.005) | below 0: 0 | below 0.3: 0 | n=300
direct T=4   0.804 ± 0.089 (SE 0.005) | below 0: 0 | below 0.3: 0 | n=300
direct T=8 (trained) 0.800 ± 0.092 (SE 0.005) | below 0: 0 | below 0.3: 0 | n=300
direct T=16  0.802 ± 0.098 (SE 0.006) | below 0: 0 | below 0.3: 1 | n=300
direct T=32  0.799 ± 0.086 (SE 0.005) | below 0: 0 | below 0.3: 0 | n=300


### 10. Spikes, operations and rough energy (same accounting as the converted SNN)
Layer 1's input current is identical at every step, so it is computed once (3.24M MACs) and only integrated T times. Every
spike in a later layer triggers an accumulate in each connection it feeds. Energy uses the commonly quoted 45 nm figures
(MAC 4.6 pJ, accumulate 0.9 pJ), as an order-of-magnitude comparison only: it ignores memory access and your fixed-point hardware.

In [12]:
n_neurons = [32 * 17 * 31, 64 * 7 * 14, 128 * 3 * 6, 512]
macs = {"conv1": 3_237_888, "conv2": 3_211_264, "conv3": 1_327_104, "fc": 1_179_648, "head": 2_048}
ann_macs = sum(macs.values())
E_MAC, E_AC = 4.6e-12, 0.9e-12

@torch.no_grad()
def spike_report(T, frames, bs=50):
    tot = np.zeros(4); n = 0
    for i in range(0, len(frames), bs):
        _, spk = net(frames[i:i + bs].to(device), T=T, return_spikes=True)
        tot += spk.sum(0).cpu().numpy(); n += len(spk)
    spk = tot / n
    sops = (spk[0] * macs["conv2"] / n_neurons[0] + spk[1] * macs["conv3"] / n_neurons[1]
            + spk[2] * macs["fc"] / n_neurons[2] + spk[3] * macs["head"] / n_neurons[3])
    energy_uJ = (macs["conv1"] * E_MAC + (sops + n_neurons[0] * T) * E_AC) * 1e6
    return spk, sops, energy_uJ

print(f"ANN: {ann_macs/1e6:.2f}M MACs, {ann_macs*E_MAC*1e6:.1f} uJ per decision\n")
spike_results = {}
for T in sorted({4, 8, 16, 32, T_TRAIN}):
    spk, sops, e = spike_report(T, eval_frames)
    rate = spk / (np.array(n_neurons) * T)
    spike_results[T] = {"spikes": spk.tolist(), "rate": rate.tolist(), "accumulates": float(sops), "energy_uJ": float(e)}
    print(f"T={T:3d}: spikes/decision {spk.round(0)} | total {spk.sum():.0f} | rate {rate.round(3)}")
    print(f"       3.24M MACs + {sops/1e6:.2f}M accumulates | est. energy {e:.1f} uJ")

ANN: 8.96M MACs, 41.2 uJ per decision

T=  4: spikes/decision [11039.  4391.   947.   562.] | total 16939 | rate [0.164 0.175 0.103 0.275]
       3.24M MACs + 3.52M accumulates | est. energy 18.1 uJ
T=  8: spikes/decision [22315.  8644.  1770.  1067.] | total 33796 | rate [0.165 0.172 0.096 0.261]
       3.24M MACs + 6.99M accumulates | est. energy 21.3 uJ
T= 16: spikes/decision [44746. 17087.  3389.  2054.] | total 67276 | rate [0.166 0.17  0.092 0.251]
       3.24M MACs + 13.88M accumulates | est. energy 27.6 uJ
T= 32: spikes/decision [89570. 33945.  6615.  4022.] | total 134152 | rate [0.166 0.169 0.09  0.245]
       3.24M MACs + 27.64M accumulates | est. energy 40.3 uJ


### 11. Comparison with the ANN and the converted SNN, and save everything

In [13]:
# reference numbers from the earlier runs (300 episodes each, same protocol)
reference = {
    "ANN (PlainCNN)":        {"mean": 0.786, "std": 0.118, "below0": 1, "note": "8.96M MACs, ~41.2 uJ"},
    "converted SNN T=8":     {"mean": 0.718, "std": 0.497, "below0": 6},
    "converted SNN T=16":    {"mean": 0.753, "std": 0.341, "below0": 4},
    "converted SNN T=32":    {"mean": 0.798, "std": 0.115, "below0": 0, "note": "3.24M MACs + 19.66M acc, ~33.1 uJ"},
    "converted SNN T=64":    {"mean": 0.791, "std": 0.110, "below0": 0, "note": "3.24M MACs + 39.13M acc, ~51.1 uJ"},
}
print(f"{'policy':24s} {'mean ± std':>16s}  below0")
for k, v in reference.items():
    print(f"{k:24s} {v['mean']:.3f} ± {v['std']:.3f}   {v['below0']:>4d}   {v.get('note', '')}")
print("-" * 70)
for T, r in game.items():
    tag = f"direct SNN T={T}" + (" *" if T == T_TRAIN else "")
    extra = ""
    if T in spike_results:
        extra = f"{spike_results[T]['accumulates']/1e6:.2f}M acc, ~{spike_results[T]['energy_uJ']:.1f} uJ"
    print(f"{tag:24s} {r.mean() if False else game[T].mean():.3f} ± {game[T].std():.3f}   {int((game[T] < 0).sum()):>4d}   {extra}")
print("(* = the T it was trained at)")

out = {
    "experiment": EXPERIMENT, "seed": SEED, "snn_T_train": T_TRAIN, "gains": gains, "target_rate": TARGET_RATE,
    "feat_scale": FEAT_SCALE, "env_steps": int(ckpt["env_steps"]), "action_repeat": ACTION_REPEAT,
    "logit_check_max_diff": float(d),
    "game": {str(T): {"mean": float(r.mean()), "std": float(r.std()), "below0": int((r < 0).sum()),
                      "below0.3": int((r < 0.3).sum()), "n": int(len(r))} for T, r in game.items()},
    "spikes": {str(k): v for k, v in spike_results.items()},
    "reference_runs": reference,
}
with open(f"{RESULTS_DIR}/{EXPERIMENT}_results.json", "w") as f:
    json.dump(out, f, indent=2)
for T, r in game.items():
    np.save(f"{RESULTS_DIR}/{EXPERIMENT}_rewards_T{T}.npy", r)
print("saved results to", RESULTS_DIR)

policy                         mean ± std  below0
ANN (PlainCNN)           0.786 ± 0.118      1   8.96M MACs, ~41.2 uJ
converted SNN T=8        0.718 ± 0.497      6   
converted SNN T=16       0.753 ± 0.341      4   
converted SNN T=32       0.798 ± 0.115      0   3.24M MACs + 19.66M acc, ~33.1 uJ
converted SNN T=64       0.791 ± 0.110      0   3.24M MACs + 39.13M acc, ~51.1 uJ
----------------------------------------------------------------------
direct SNN T=2           0.804 ± 0.090      0   
direct SNN T=4           0.804 ± 0.089      0   3.52M acc, ~18.1 uJ
direct SNN T=8 *         0.800 ± 0.092      0   6.99M acc, ~21.3 uJ
direct SNN T=16          0.802 ± 0.098      0   13.88M acc, ~27.6 uJ
direct SNN T=32          0.799 ± 0.086      0   27.64M acc, ~40.3 uJ
(* = the T it was trained at)
saved results to /home/e22405/doom-on-snn/GPU folders/outputs/direct_snn_results


### How to read the result, and what to run next

- **Reward at the trained T versus the converted SNN.** The question this notebook answers: does training the spiking network directly reach the
  ANN's reward at a smaller T than the converted SNN's T=32? Compare means, spread (failed episodes drive the spread at small T) and the
  operation counts. One seed is not enough to conclude anything: repeat with `SEED = 1, 2` (and the same seeds for the converted SNN).
- **Reward at other T.** The network was trained at one T. Rows at other T show how sensitive it is to running for more or fewer steps.
- **If it does not learn.** Compare the training log with the ANN run (the ANN passed 0.75 within about 2M steps). Then try `FEAT_SCALE`,
  the learning rate, and `TARGET_RATE`, one at a time, and report them as changes. A failed seed is a result too.
- **Fair comparison caveats.** Same architecture, steps, protocol and operation accounting, but a directly trained SNN has fixed gains and a
  feature scale that the ANN does not have, and its training cost is roughly T times higher per update.
- **Smoke test first.** With `SMOKE_TEST = True` the run takes 100k steps. Check that the reward moves above the random-policy level and note the training speed
  before starting the 2M-step run.